In [ ]:
import os
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, cohen_kappa_score

# ==========================================
# [TASK 3] ATTENTION MECHANISM (CBAM)
# Definition of Channel and Spatial Attention
# ==========================================
class ChannelAttention(nn.Module):
    def __init__(self, in_planes, ratio=16):
        super(ChannelAttention, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.fc1 = nn.Conv2d(in_planes, in_planes // ratio, 1, bias=False)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Conv2d(in_planes // ratio, in_planes, 1, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = self.fc2(self.relu1(self.fc1(self.avg_pool(x))))
        max_out = self.fc2(self.relu1(self.fc1(self.max_pool(x))))
        out = avg_out + max_out
        return self.sigmoid(out)

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super(SpatialAttention, self).__init__()
        assert kernel_size in (3, 7), 'kernel size must be 3 or 7'
        padding = 3 if kernel_size == 7 else 1
        self.conv1 = nn.Conv2d(2, 1, kernel_size, padding=padding, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        x = torch.cat([avg_out, max_out], dim=1)
        x = self.conv1(x)
        return self.sigmoid(x)

class CBAM(nn.Module):
    def __init__(self, in_planes, ratio=16, kernel_size=7):
        super(CBAM, self).__init__()
        self.ca = ChannelAttention(in_planes, ratio)
        self.sa = SpatialAttention(kernel_size)

    def forward(self, x):
        out = x * self.ca(x)
        result = out * self.sa(out)
        return result

# ==========================================
# DATASET PREPARATION
# ==========================================
class RetinaMultiLabelDataset(Dataset):
    def __init__(self, csv_file, image_dir, transform=None):
        self.data = pd.read_csv(csv_file)
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        img_name = row.iloc[0]
        img_path = os.path.join(self.image_dir, img_name)
        
        try:
            img = Image.open(img_path).convert("RGB")
        except FileNotFoundError:
            img = Image.new('RGB', (256, 256))
        
        # Labels for D, G, A
        labels = torch.tensor(row[1:4].values.astype("float32")) 
        
        if self.transform:
            img = self.transform(img)
            
        return img, labels, img_name

# ==========================================
# MODEL BUILDING
# Includes [TASK 1] (Backbone) and [TASK 3] (Attention Injection)
# ==========================================
def build_model(backbone="resnet18", num_classes=3, pretrained=True, use_attention=True):
    if backbone == "resnet18":
        # [TASK 1] Initialize ResNet18 Backbone
        model = models.resnet18(pretrained=pretrained)
        
        # [TASK 3] Insert CBAM Attention Modules
        # We add them as new modules to the existing sequential blocks
        if use_attention:
            model.layer1.add_module("cbam", CBAM(64))
            model.layer2.add_module("cbam", CBAM(128))
            model.layer3.add_module("cbam", CBAM(256))
            model.layer4.add_module("cbam", CBAM(512))

        # [TASK 1] Fine-tuning the classifier for 3 classes
        model.fc = nn.Linear(model.fc.in_features, num_classes)
        
    elif backbone == "efficientnet":
        model = models.efficientnet_b0(pretrained=pretrained)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
    else:
        raise ValueError("Unsupported backbone")
    return model

# ==========================================
# TRAINING LOOP
# Includes [TASK 2] (Loss) and [TASK 1] (Weight Loading)
# ==========================================
def train_one_backbone(backbone, train_csv, val_csv, test_csv, onsite_csv,
                       train_image_dir, val_image_dir, test_image_dir, onsite_image_dir,
                       epochs=10, batch_size=32, lr=1e-4, img_size=256, save_dir="checkpoints", 
                       pretrained_backbone=None):
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    # Transforms
    transform = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
    ])
    
    val_transform = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
    ])

    # Datasets
    train_ds = RetinaMultiLabelDataset(train_csv, train_image_dir, transform)
    val_ds   = RetinaMultiLabelDataset(val_csv, val_image_dir, val_transform)
    test_ds  = RetinaMultiLabelDataset(test_csv, test_image_dir, val_transform)
    onsite_ds = RetinaMultiLabelDataset(onsite_csv, onsite_image_dir, val_transform)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2)
    val_loader   = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=2)
    test_loader  = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=2)
    onsite_loader = DataLoader(onsite_ds, batch_size=batch_size, shuffle=False, num_workers=2)

    # Initialize Model
    # We pass 'pretrained=False' here because we load YOUR specific backbone file later
    model = build_model(backbone, num_classes=3, pretrained=False, use_attention=True).to(device)

    for p in model.parameters():
        p.requires_grad = True
    
    # ---------------------------------------------------------
    # [TASK 2] CLASS-BALANCED LOSS FUNCTION
    # ---------------------------------------------------------
    print(">>> [TASK 2] Computing class weights for Balanced Loss...")
    df_train = pd.read_csv(train_csv)
    labels = df_train.iloc[:, 1:4].values # Columns D, G, A
    pos_counts = np.sum(labels, axis=0)
    neg_counts = len(labels) - pos_counts
    # Formula: Weight = Negatives / Positives
    pos_weights = torch.tensor(neg_counts / (pos_counts + 1e-6), dtype=torch.float32).to(device)
    print(f"    Class Weights (D, G, A): {pos_weights.cpu().numpy()}")
    
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weights)
    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)

    # ---------------------------------------------------------
    # [TASK 1] LOADING PRETRAINED BACKBONE (TRANSFER LEARNING)
    # ---------------------------------------------------------
    if pretrained_backbone is not None and os.path.exists(pretrained_backbone):
        print(f">>> [TASK 1] Loading pretrained backbone from {pretrained_backbone}")
        try:
            state_dict = torch.load(pretrained_backbone, map_location=device)
            if 'state_dict' in state_dict:
                state_dict = state_dict['state_dict']
            
            # strict=False is required because our model has new Attention layers 
            # and a different FC layer than the backbone
            missing, unexpected = model.load_state_dict(state_dict, strict=False)
            print(f"    Backbone loaded successfully.")
        except Exception as e:
            print(f"    Error loading backbone: {e}")
    else:
        print("    Warning: Pretrained backbone not found.")

    # Training Loop
    best_val_loss = float("inf")
    os.makedirs(save_dir, exist_ok=True)
    ckpt_path = os.path.join(save_dir, f"best_{backbone}.pt")

    for epoch in range(epochs):
        model.train()
        train_loss = 0
        
        loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]", leave=False)
        for imgs, labels, _ in loop:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * imgs.size(0)
            loop.set_postfix(loss=loss.item())

        train_loss /= len(train_loader.dataset)

        # Validation
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for imgs, labels, _ in val_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                outputs = model(imgs)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * imgs.size(0)
        val_loss /= len(val_loader.dataset)

        print(f"[{backbone}] Epoch {epoch+1}/{epochs} Train Loss: {train_loss:.4f} Val Loss: {val_loss:.4f}")

        # Save Best Model
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), ckpt_path)
            print(f"    Saved best model: {ckpt_path}")

    # ========================
    # Evaluation on Offsite Test
    # ========================
    print("\n>>> Evaluating on Offsite Test Set...")
    if os.path.exists(ckpt_path):
        model.load_state_dict(torch.load(ckpt_path, map_location=device))
    
    model.eval()
    y_true, y_pred = [], []

    with torch.no_grad():
        for imgs, labels, _ in tqdm(test_loader, desc="Offsite Testing"):
            imgs = imgs.to(device)
            outputs = model(imgs)
            probs = torch.sigmoid(outputs).cpu().numpy()
            preds = (probs > 0.5).astype(int)
            y_true.extend(labels.numpy())
            y_pred.extend(preds)

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    disease_names = ["DR", "Glaucoma", "AMD"]
    print("-" * 30)
    for i, disease in enumerate(disease_names):
        y_t = y_true[:, i]
        y_p = y_pred[:, i]
        acc = accuracy_score(y_t, y_p)
        precision = precision_score(y_t, y_p, average="macro", zero_division=0)
        recall = recall_score(y_t, y_p, average="macro", zero_division=0)
        f1 = f1_score(y_t, y_p, average="macro", zero_division=0)
        kappa = cohen_kappa_score(y_t, y_p)
        print(f"{disease} | Acc: {acc:.4f} | Prec: {precision:.4f} | Rec: {recall:.4f} | F1: {f1:.4f} | Kappa: {kappa:.4f}")
    print("-" * 30)

    # ========================
    # Inference on Onsite Test (For Submission)
    # ========================
    print("\n>>> Generating Submission for Onsite Test Set...")
    onsite_results = []
    
    model.eval()
    with torch.no_grad():
        for imgs, _, img_names in tqdm(onsite_loader, desc="Onsite Inference"):
            imgs = imgs.to(device)
            outputs = model(imgs)
            probs = torch.sigmoid(outputs).cpu().numpy()
            preds = (probs > 0.5).astype(int)
            
            for i in range(len(img_names)):
                row = [img_names[i], preds[i][0], preds[i][1], preds[i][2]]
                onsite_results.append(row)
    
    submission_df = pd.DataFrame(onsite_results, columns=['id', 'D', 'G', 'A'])
    submission_df.to_csv("submission.csv", index=False)
    print(">>> 'submission.csv' saved successfully!")

# ========================
# Main
# ========================
if __name__ == "__main__":
    train_csv = "train.csv" 
    val_csv   = "val.csv" 
    test_csv  = "offsite_test.csv" 
    onsite_csv = "onsite_test_submission.csv"
    
    # Check your Colab folder paths!
    train_image_dir ="./Images/train" 
    val_image_dir = "./Images/val"
    test_image_dir = "./Images/offsite_test"
    onsite_image_dir = "./Images/onsite_test"
    
    pretrained_backbone = './pretrained_backbone/ckpt_resnet18_ep50.pt'
    backbone = 'resnet18' 
    
    train_one_backbone(backbone, train_csv, val_csv, test_csv, onsite_csv,
                       train_image_dir, val_image_dir, test_image_dir, onsite_image_dir,
                       epochs=15, batch_size=32, lr=1e-4, img_size=256, pretrained_backbone=pretrained_backbone)